In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm


# Load data

df = pd.read_excel("business_regression_data.xlsx")


# Simple Model 1
# monthly_sales ~ marketing_spend

X1 = sm.add_constant(df[['marketing_spend']])
y = df['monthly_sales']

data1 = pd.concat([X1, y], axis=1).dropna()
X1 = data1.drop(columns=['monthly_sales'])
y1 = data1['monthly_sales']

model1 = sm.OLS(y1, X1).fit()


# Simple Model 2
# monthly_sales ~ footfall

X2 = sm.add_constant(df[['footfall']])

data2 = pd.concat([X2, y], axis=1).dropna()
X2 = data2.drop(columns=['monthly_sales'])
y2 = data2['monthly_sales']

model2 = sm.OLS(y2, X2).fit()


# Multiple Regression

df_model = pd.get_dummies(
    df,
    columns=['region','store_type'],
    drop_first=True,
    dtype=int
)

predictors = [
    'marketing_spend',
    'footfall',
    'inventory_availability_pct',
    'customer_rating'
]

dummy_cols = [c for c in df_model.columns
               if c.startswith('region_') or c.startswith('store_type_')]

predictors += dummy_cols

X3 = df_model[predictors]
X3 = X3.replace([np.inf,-np.inf], np.nan)

combined = pd.concat([X3, df_model['monthly_sales']], axis=1).dropna()

X3 = combined.drop(columns=['monthly_sales'])
y3 = combined['monthly_sales']

X3 = sm.add_constant(X3)

model3 = sm.OLS(y3, X3).fit()


# Model Comparison Table

comparison_df = pd.DataFrame({
    "Model_Name":[
        "Simple Regression - Marketing Spend",
        "Simple Regression - Footfall",
        "Multiple Regression"
    ],
    
    "Variables_Used":[
        "marketing_spend",
        "footfall",
        "marketing_spend, footfall, inventory_availability_pct, customer_rating + dummy variables"
    ],
    
    "R_squared":[
        model1.rsquared,
        model2.rsquared,
        model3.rsquared
    ],
    
    "Business_Usefulness":[
        "Explains impact of marketing spend",
        "Explains impact of footfall",
        "Most useful for decision making"
    ]
})

print(comparison_df)


# Save regression_summary.xlsx

comparison_df.to_excel(
    "regression_summary.xlsx",
    index=False
)


# Update Workbook

with pd.ExcelWriter(
        "regression_workbook.xlsx",
        engine="openpyxl",
        mode="a",
        if_sheet_exists="replace"
) as writer:
    
    comparison_df.to_excel(
        writer,
        sheet_name="Model_Comparison",
        index=False
    )

print("\nregression_summary.xlsx created successfully.")

                            Model_Name  \
0  Simple Regression - Marketing Spend   
1         Simple Regression - Footfall   
2                  Multiple Regression   

                                      Variables_Used  R_squared  \
0                                    marketing_spend   0.167245   
1                                           footfall   0.736285   
2  marketing_spend, footfall, inventory_availabil...   0.827755   

                  Business_Usefulness  
0  Explains impact of marketing spend  
1         Explains impact of footfall  
2     Most useful for decision making  

regression_summary.xlsx created successfully.
